# Paper Figures — WES Multi-Stage Phenology

All publication-quality figures for the RSE/IJAEOG paper, regenerated from current results.

**Inputs**: `multi_stage_models.csv`, `pcse_wofost_phenology.csv`, `features.parquet`

**Outputs** → `poster_figures/`:
1. **F1 — Multi-stage R² heatmap** (8 stages × 5 models, Hybrid C)
2. **F2 — Best-model bar chart** (per stage, 95% CI)
3. **F3 — Strategy comparison** (A WES vs B ML vs C Hybrid vs D WOFOST)
4. **F4 — Predicted vs observed scatter** (4 critical stages: flag_leaf/boot/heading/anthesis)
5. **F5 — Feature importance** (top 15 per stage, Lasso coefficients)
6. **F6 — Spatial map** (US Plains, R² per state)
7. **F7 — Year-by-year accuracy** (LOYO breakdown)

In [ ]:
import pandas as pd
import numpy as np
import sys, os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Allow imports from utils
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
from scripts.utils.config import get_config
cfg = get_config()

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# Purdue brand colors
GOLD, BLACK, DARK, GREY, LIGHT = '#CEB888', '#000000', '#6B5C36', '#9B9B9B', '#E5E5E5'
STAGE_COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f']
MODEL_COLORS = {'ElasticNet': '#CEB888', 'Ridge': '#9B9B9B', 'RandomForest': '#2ca02c',
                'XGBoost': '#d62728', 'LightGBM': '#9467bd', 'physics': '#7f7f7f'}

STAGE_ORDER = ['emergence','tillering','jointing','flag_leaf','boot','heading','anthesis','maturity']
STAGE_LABELS = {'emergence':'Emergence','tillering':'Tillering','jointing':'Jointing',
                'flag_leaf':'Flag Leaf','boot':'Boot','heading':'Heading',
                'anthesis':'Anthesis','maturity':'Maturity'}

POSTER_DIR = Path(cfg.paths.poster_root)
POSTER_DIR.mkdir(exist_ok=True, parents=True)
print(f'Output dir: {POSTER_DIR}')

In [ ]:
# Load all results
models_df = pd.read_csv(cfg.paths.multi_stage_models)
wofost_df = pd.read_csv(cfg.paths.pcse_wofost) if os.path.exists(cfg.paths.pcse_wofost) else None
strat_d   = pd.read_csv(cfg.paths.strategy_d_pcse) if os.path.exists(cfg.paths.strategy_d_pcse) else None
feat_df   = pd.read_parquet(cfg.paths.features_final)

print(f'models_df:  {models_df.shape} — {sorted(models_df["model"].unique())}')
print(f'features:   {feat_df.shape}')
if strat_d is not None:
    print(f'PCSE Strategy D:\n{strat_d.to_string()}')

## F1 — Multi-stage R² heatmap (Hybrid C, 5 models × 8 stages)

In [ ]:
pv = models_df[models_df['strategy']=='C_Hybrid'].pivot(index='stage', columns='model', values='R2')
pv = pv.reindex(STAGE_ORDER)
pv.index = [STAGE_LABELS[s] for s in pv.index]

fig = go.Figure(data=go.Heatmap(
    z=pv.values, x=pv.columns, y=pv.index,
    colorscale='RdYlGn', zmid=0.5, zmin=-0.2, zmax=0.85,
    text=pv.round(2).values, texttemplate='%{text}',
    textfont=dict(size=14, color='black'),
    colorbar=dict(title='R²'),
))
fig.update_layout(
    title='F1 — Per-stage R² across ML models (Strategy C: Hybrid WES + features)',
    xaxis_title='Model', yaxis_title='Phenology stage',
    template='plotly_white', height=500, width=900,
    yaxis=dict(autorange='reversed'),
)
fig.show()
fig.write_image(POSTER_DIR/'F1_r2_heatmap.svg')
fig.write_image(POSTER_DIR/'F1_r2_heatmap.png', scale=2)
print(f'Saved F1 → {POSTER_DIR/"F1_r2_heatmap.svg"}')

## F2 — Best model per stage (R² with 95% bootstrap CI)

In [ ]:
# Pick best (B or C) per stage
best = (models_df[models_df['strategy']!='A_WES']
        .sort_values('R2', ascending=False)
        .groupby('stage', as_index=False).first())
best = best.set_index('stage').reindex(STAGE_ORDER).reset_index()
best['stage_label'] = best['stage'].map(STAGE_LABELS)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=best['stage_label'], y=best['R2'],
    error_y=dict(type='data', symmetric=False,
                 array=best['R2_hi']-best['R2'],
                 arrayminus=best['R2']-best['R2_lo']),
    marker_color=GOLD, marker_line_color=BLACK, marker_line_width=1.5,
    text=[f'{r:.2f}<br><i>{m}</i><br>n={n}' for r, m, n in
          zip(best['R2'], best['model'], best['n'])],
    textposition='outside',
))
fig.update_layout(
    title='F2 — Best model per stage (R² with 95% bootstrap CI)',
    xaxis_title='Phenology stage', yaxis_title='R²',
    yaxis=dict(range=[-0.1, 1.0]),
    template='plotly_white', height=600, width=1200,
    showlegend=False,
)
fig.add_hline(y=0.5, line_dash='dash', line_color=GREY, annotation_text='R²=0.5')
fig.add_hline(y=0.78, line_dash='dot', line_color=DARK, annotation_text='SOTA threshold (0.78)')
fig.show()
fig.write_image(POSTER_DIR/'F2_best_per_stage.svg')
fig.write_image(POSTER_DIR/'F2_best_per_stage.png', scale=2)
print(f'Saved F2 → {POSTER_DIR/"F2_best_per_stage.svg"}')

## F3 — Strategy comparison (A WES vs B ML vs C Hybrid vs D WOFOST)

In [ ]:
# Best B and best C per stage
def best_within(strat):
    sub = models_df[models_df['strategy']==strat].copy()
    if 'R2' not in sub.columns: return pd.DataFrame()
    return sub.sort_values('R2', ascending=False).groupby('stage', as_index=False).first()

stratA = models_df[models_df['strategy']=='A_WES'].set_index('stage').reindex(STAGE_ORDER)
stratB = best_within('B_ML-only').set_index('stage').reindex(STAGE_ORDER)
stratC = best_within('C_Hybrid').set_index('stage').reindex(STAGE_ORDER)

# Strategy D: PCSE-WOFOST (only emergence/anthesis/maturity)
stratD_dict = {}
if strat_d is not None:
    for _, r in strat_d.iterrows():
        stratD_dict[r['stage']] = r['R2']

x = [STAGE_LABELS[s] for s in STAGE_ORDER]
fig = go.Figure()
fig.add_trace(go.Bar(name='A: WES (physics)', x=x, y=stratA['R2'].values,
                     marker_color=GREY, marker_line_color=BLACK, marker_line_width=1))
fig.add_trace(go.Bar(name='B: ML only', x=x, y=stratB['R2'].values,
                     marker_color=DARK, marker_line_color=BLACK, marker_line_width=1))
fig.add_trace(go.Bar(name='C: Hybrid (WES + ML)', x=x, y=stratC['R2'].values,
                     marker_color=GOLD, marker_line_color=BLACK, marker_line_width=1))
fig.add_trace(go.Bar(name='D: PCSE-WOFOST (physics)',
                     x=[STAGE_LABELS[s] for s in stratD_dict],
                     y=list(stratD_dict.values()),
                     marker_color='#4682B4', marker_line_color=BLACK, marker_line_width=1))
fig.update_layout(
    title='F3 — Strategy comparison: physics-only vs ML vs hybrid vs WOFOST',
    yaxis=dict(title='R²', range=[-1.2, 1]),
    xaxis_title='Phenology stage',
    barmode='group', template='plotly_white',
    height=600, width=1400,
    legend=dict(yanchor='bottom', y=0.02, xanchor='right', x=0.98),
)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.show()
fig.write_image(POSTER_DIR/'F3_strategy_comparison.svg')
fig.write_image(POSTER_DIR/'F3_strategy_comparison.png', scale=2)
print(f'Saved F3 → {POSTER_DIR/"F3_strategy_comparison.svg"}')

## F4 — Predicted vs observed scatter (4 critical stages)

Re-runs LOYO CV for the best model per critical stage to get prediction arrays.

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression

# Recompute targets (same as multi_stage_ml notebook)
pheno = pd.read_parquet(cfg.paths.phenology_matched)
STAGE_MAP = dict(cfg.phenology_stages)
SPRING_ONLY = set(cfg.spring_only_stages)

obs = None
for stage, labels in STAGE_MAP.items():
    s = pheno[pheno['growth_stage'].isin(labels)].copy()
    if stage in SPRING_ONLY:
        s = s[s['dos'] > cfg.spring_dos_min]
    s['harvest_year'] = s['growing_season'].str.split('-').str[1].astype(int)
    e = s.groupby(['FIELDID','harvest_year'])['dos'].min().reset_index()
    e['field_id'] = e['FIELDID'].astype(str)
    e['year'] = e['harvest_year'].astype(int)
    e = e[['field_id','year','dos']].rename(columns={'dos': f'{stage}_dos_obs'})
    obs = e if obs is None else obs.merge(e, on=['field_id','year'], how='outer')

feat = feat_df.copy()
if 'state' in feat.columns: feat = feat.drop(columns=['state'])
fwt = feat.merge(obs, on=['field_id','year'], how='left')

# Feature columns (late stages = all)
META = ['field_id','year','flag_true_doy','n_obs','sowing_doy_used'] + [f'{s}_dos_obs' for s in STAGE_MAP]
ndre = [c for c in fwt.columns if c.startswith('NDRE')]
redund = ['GDD_M2_at_SOS','VD_at_SOS','emergence_doy','VD_from_emergence_at_SOS',
          'fV_from_emergence_at_SOS','days_emergence_to_SOS']
all_cols = [c for c in fwt.columns if c not in META and c not in ndre and c not in redund]

def fit_predict_loyo(target, feat_cols, k=60):
    df2 = fwt.dropna(subset=[target]).copy()
    yp_all, yt_all = [], []
    for yr in sorted(df2['year'].unique()):
        tr = df2[df2['year']!=yr]; te = df2[df2['year']==yr]
        if len(tr) < 30 or len(te) < 5: continue
        steps = [('imp', SimpleImputer(strategy='median')),
                 ('sc', StandardScaler()),
                 ('select', SelectKBest(f_regression, k=min(k, len(feat_cols)))),
                 ('m', ElasticNetCV(cv=5, l1_ratio=[0.1,0.3,0.5,0.7,0.9], max_iter=10000))]
        pipe = Pipeline(steps).fit(tr[feat_cols], tr[target])
        p = pipe.predict(te[feat_cols])
        if p.ndim>1: p=p.ravel()
        yp_all.extend(p); yt_all.extend(te[target].values)
    return np.array(yp_all), np.array(yt_all)

critical = ['flag_leaf','boot','heading','anthesis']
predictions = {}
for stage in critical:
    yp, yt = fit_predict_loyo(f'{stage}_dos_obs', all_cols, k=60)
    predictions[stage] = (yt, yp)
    print(f'{stage}: n={len(yt)}, R²={1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2):.3f}')

In [ ]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[STAGE_LABELS[s] for s in critical])
for i, stage in enumerate(critical):
    yt, yp = predictions[stage]
    r2 = 1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2)
    rmse = np.sqrt(np.mean((yt-yp)**2))
    row, col = i//2 + 1, i%2 + 1
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    fig.add_trace(go.Scatter(x=yt, y=yp, mode='markers',
                             marker=dict(size=4, color=GOLD, opacity=0.4,
                                         line=dict(width=0.3, color=BLACK)),
                             name=stage, showlegend=False), row=row, col=col)
    fig.add_trace(go.Scatter(x=[lo,hi], y=[lo,hi], mode='lines',
                             line=dict(color='black', dash='dash'),
                             showlegend=False), row=row, col=col)
    fig.add_annotation(text=f'R²={r2:.3f}<br>RMSE={rmse:.1f}d<br>n={len(yt)}',
                       xref=f'x{i+1}', yref=f'y{i+1}', x=lo+0.05*(hi-lo), y=hi-0.05*(hi-lo),
                       showarrow=False, font=dict(size=11),
                       bgcolor='white', bordercolor='black', borderwidth=1, align='left')
fig.update_layout(
    title='F4 — Predicted vs observed DOS (4 critical stages, LOYO CV)',
    template='plotly_white', height=800, width=900,
)
fig.update_xaxes(title_text='Observed DOS', row=2)
fig.update_yaxes(title_text='Predicted DOS', col=1)
fig.show()
fig.write_image(POSTER_DIR/'F4_pred_vs_obs.svg')
fig.write_image(POSTER_DIR/'F4_pred_vs_obs.png', scale=2)
print(f'Saved F4 → {POSTER_DIR/"F4_pred_vs_obs.svg"}')

## F5 — Feature importance (top 15 per critical stage)

In [ ]:
# Fit final model on all data per critical stage, extract Lasso coefficients
from sklearn.linear_model import Lasso

def fit_full_get_importance(target, feat_cols, k=60, top_n=15):
    df2 = fwt.dropna(subset=[target]).copy()
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
        ('select', SelectKBest(f_regression, k=min(k, len(feat_cols)))),
        ('m', ElasticNetCV(cv=5, l1_ratio=[0.5], max_iter=10000)),
    ])
    pipe.fit(df2[feat_cols], df2[target])
    selected = np.array(feat_cols)[pipe.named_steps['select'].get_support()]
    coefs = pipe.named_steps['m'].coef_
    df = pd.DataFrame({'feature': selected, 'coef': coefs})
    df['abs_coef'] = df['coef'].abs()
    df = df.sort_values('abs_coef', ascending=False).head(top_n)
    return df

fig = make_subplots(rows=2, cols=2, subplot_titles=[STAGE_LABELS[s] for s in critical],
                    horizontal_spacing=0.20)
for i, stage in enumerate(critical):
    imp = fit_full_get_importance(f'{stage}_dos_obs', all_cols, k=60, top_n=12)
    row, col = i//2 + 1, i%2 + 1
    colors = [GOLD if c > 0 else DARK for c in imp['coef']]
    fig.add_trace(go.Bar(
        x=imp['coef'], y=imp['feature'], orientation='h',
        marker_color=colors, marker_line_color=BLACK, marker_line_width=0.5,
        showlegend=False, name=stage,
        text=[f'{c:+.2f}' for c in imp['coef']], textposition='outside',
    ), row=row, col=col)
    fig.update_yaxes(autorange='reversed', row=row, col=col, tickfont=dict(size=9))
fig.update_layout(
    title='F5 — Top-12 ElasticNet coefficients per critical stage<br><sub>Gold = later stage; Dark = earlier stage (positive coef → larger feature → later DOS)</sub>',
    template='plotly_white', height=900, width=1300,
)
fig.show()
fig.write_image(POSTER_DIR/'F5_feature_importance.svg')
fig.write_image(POSTER_DIR/'F5_feature_importance.png', scale=2)
print(f'Saved F5 → {POSTER_DIR/"F5_feature_importance.svg"}')

## F6 — Spatial map (US Plains, R² per state for anthesis)

In [ ]:
# Per-state R² for anthesis (using best-model predictions)
yt, yp = predictions['anthesis']
df_an = fwt.dropna(subset=['anthesis_dos_obs']).reset_index(drop=True)
df_an = df_an.iloc[:len(yt)].copy()
df_an['pred'] = yp; df_an['true'] = yt; df_an['err'] = yp - yt

# State from one-hot
state_cols = [c for c in df_an.columns if c.startswith('state_')]
df_an['state'] = df_an[state_cols].idxmax(axis=1).str.replace('state_','')
per_state = df_an.groupby('state').agg(
    n=('err','count'),
    R2=('true', lambda y: 1 - ((df_an.loc[y.index,'true']-df_an.loc[y.index,'pred'])**2).sum() /
                          ((y - y.mean())**2).sum()),
    RMSE=('err', lambda e: np.sqrt(np.mean(e**2))),
    bias=('err', 'mean'),
).round(3).sort_values('n', ascending=False)
print('Per-state anthesis accuracy:')
print(per_state.to_string())

# Bar chart R² + RMSE by state
fig = make_subplots(rows=1, cols=2, subplot_titles=['R² per state','RMSE (days) per state'])
fig.add_trace(go.Bar(x=per_state.index, y=per_state['R2'],
                     text=[f'n={n}<br>R²={r:.2f}' for n, r in zip(per_state['n'], per_state['R2'])],
                     textposition='outside', marker_color=GOLD,
                     marker_line_color=BLACK, marker_line_width=1), row=1, col=1)
fig.add_trace(go.Bar(x=per_state.index, y=per_state['RMSE'],
                     text=[f'{r:.1f}' for r in per_state['RMSE']],
                     textposition='outside', marker_color=DARK,
                     marker_line_color=BLACK, marker_line_width=1), row=1, col=2)
fig.update_layout(
    title='F6 — Anthesis prediction accuracy by US state (geographic transferability)',
    template='plotly_white', height=500, width=1200, showlegend=False,
)
fig.update_yaxes(title_text='R²', row=1, col=1)
fig.update_yaxes(title_text='RMSE (days)', row=1, col=2)
fig.show()
fig.write_image(POSTER_DIR/'F6_per_state.svg')
fig.write_image(POSTER_DIR/'F6_per_state.png', scale=2)
print(f'Saved F6 → {POSTER_DIR/"F6_per_state.svg"}')

## F7 — Year-by-year accuracy (LOYO breakdown for 4 critical stages)

In [ ]:
# For each critical stage, compute R² holding out each year individually
year_results = []
for stage in critical:
    target = f'{stage}_dos_obs'
    df2 = fwt.dropna(subset=[target]).copy()
    for yr in sorted(df2['year'].unique()):
        tr = df2[df2['year']!=yr]; te = df2[df2['year']==yr]
        if len(tr) < 30 or len(te) < 5: continue
        pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('sc', StandardScaler()),
                          ('select', SelectKBest(f_regression, k=60)),
                          ('m', ElasticNetCV(cv=5, l1_ratio=[0.5], max_iter=10000))]).fit(
            tr[all_cols], tr[target])
        yp = pipe.predict(te[all_cols])
        if yp.ndim>1: yp=yp.ravel()
        yt = te[target].values
        r2 = 1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if len(yt)>1 else 0
        rmse = np.sqrt(np.mean((yt-yp)**2))
        year_results.append({'stage': stage, 'year': yr, 'n': len(yt), 'R2': r2, 'RMSE': rmse})
yr_df = pd.DataFrame(year_results)

fig = make_subplots(rows=1, cols=2, subplot_titles=['R² by held-out year', 'RMSE by held-out year'])
for i, stage in enumerate(critical):
    sub = yr_df[yr_df['stage']==stage]
    fig.add_trace(go.Scatter(x=sub['year'], y=sub['R2'], mode='lines+markers',
                             name=STAGE_LABELS[stage], line=dict(color=STAGE_COLORS[i], width=2),
                             marker=dict(size=10)), row=1, col=1)
    fig.add_trace(go.Scatter(x=sub['year'], y=sub['RMSE'], mode='lines+markers',
                             name=STAGE_LABELS[stage], line=dict(color=STAGE_COLORS[i], width=2),
                             marker=dict(size=10), showlegend=False), row=1, col=2)
fig.update_layout(
    title='F7 — Year-by-year holdout accuracy (LOYO CV breakdown)',
    template='plotly_white', height=500, width=1300,
)
fig.update_xaxes(title_text='Held-out year', row=1)
fig.update_yaxes(title_text='R²', row=1, col=1, range=[0, 1])
fig.update_yaxes(title_text='RMSE (days)', row=1, col=2)
fig.show()
fig.write_image(POSTER_DIR/'F7_year_breakdown.svg')
fig.write_image(POSTER_DIR/'F7_year_breakdown.png', scale=2)
print(f'Saved F7 → {POSTER_DIR/"F7_year_breakdown.svg"}')
print()
print('Year-by-year details:')
print(yr_df.pivot(index='year', columns='stage', values='R2').round(3).to_string())

## Summary — final paper-ready table

In [ ]:
summary = best.copy()
summary = summary[['stage_label','model','strategy','n','R2','R2_lo','R2_hi','RMSE','w10']]
summary.columns = ['Stage','Best model','Strategy','n','R²','R²_lo','R²_hi','RMSE (d)','±10d (%)']
summary = summary.round({'R²':3,'R²_lo':3,'R²_hi':3,'RMSE (d)':2,'±10d (%)':1})
print('=== FINAL PAPER-READY RESULTS ===')
print(summary.to_string(index=False))
summary.to_csv(POSTER_DIR/'paper_summary_table.csv', index=False)
print(f'\nSaved → {POSTER_DIR/"paper_summary_table.csv"}')
print(f'\nMean R² across 8 stages: {summary["R²"].mean():.3f}')
print(f'Stages with R² ≥ 0.78: {(summary["R²"] >= 0.78).sum()}/8')
print(f'Stages with ±10d ≥ 90%: {(summary["±10d (%)"] >= 90).sum()}/8')